# Map EV charging stations to different regions

## https://overpass-turbo.eu/ is used

In [1]:
import pandas as pd
import numpy as np
df_charging_stations = pd.read_csv('./UK_charging_stations_with_capacity.csv')
# df_charging_stations = df_charging_stations[df_charging_stations["NUTS_NAME"].notna()]

df_charging_stations

,StationID,Title,AddressLine1,Town,Latitude,Longitude,UsageCost,Status,PowerKW,Voltage,Amps,CurrentType,ConnectionType,Level,Quantity
0,459890,Badger Inn,Church Minshull,Nantwich,53.140998,-2.500894,NaN,Operational,7.0,230.0,32.0,AC (Single-Phase),Type 2 (Socket Only),Level 2 : Medium (Over 2kW),2.0
1,459857,"Elmswell Services (EV Tesla OTM) - Elmswell, UK",15 A1088,Bury Saint Edmunds,52.231449,0.893506,£0.49/kWh;other tariffs available,Operational,250.0,NaN,NaN,DC,CCS (Type 2),Level 3: High (Over 40kW),8.0
2,459397,North East Garages,Claremont Crescent,Whitley Bay,55.054844,-1.462634,33p via Octopus Card,Operational,22.0,NaN,NaN,AC (Three-Phase),Type 2 (Socket Only),Level 2 : Medium (Over 2kW),1.0
3,459397,North East Garages,Claremont Crescent,Whitley Bay,55.054844,-1.462634,33p via Octopus Card,Operational,22.0,NaN,NaN,AC (Three-Phase),Type 2 (Socket Only),Level 2 : Medium (Over 2kW),1.0
4,459309,Anglesey Supercharger,Menai,Gaerwen,53.223435,-4.264561,NaN,Operational,250.0,NaN,NaN,DC,CCS (Type 2),Level 3: High (Over 40kW),12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47184,4124,Hurst Nissan Centre Belfast,62 Boucher Road,Belfast,54.571881,-5.968309,Free,Operational,NaN,NaN,NaN,AC (Single-Phase),Type 2 (Socket Only),Level 2 : Medium (Over 2kW),1.0
47185,4123,Alex F Noble & Sons Nissan,Swinton Place,Edinburgh,55.875053,-3.173333,Free,Operational,50.0,400.0,125.0,AC (Three-Phase),CHAdeMO,Level 3: High (Over 40kW),1.0
47186,4123,Alex F Noble & Sons Nissan,Swinton Place,Edinburgh,55.875053,-3.173333,Free,Operational,7.0,230.0,32.0,AC (Single-Phase),Type 2 (Socket Only),Level 2 : Medium (Over 2kW),2.0
47187,4121,Union Square Car Park,Union Square,Aberdeen,57.142000,-2.096000,Free but Parking charges apply,Operational,7.0,230.0,32.0,AC (Single-Phase),Type 2 (Socket Only),Level 2 : Medium (Over 2kW),10.0


In [2]:
from shapely.geometry import Point
import geopandas as gpd
# Load the regions GeoDataFrame
regions = gpd.read_file('./ref-nuts-2021-01m.geojson/NUTS_RG_01M_2021_4326_LEVL_3.geojson')

geometry_charging_stations = [Point(xy) for xy in zip(df_charging_stations['Longitude'], df_charging_stations['Latitude'])]
charging_gdf = gpd.GeoDataFrame(df_charging_stations, geometry=geometry_charging_stations, crs="EPSG:4326")

charging_gdf = charging_gdf.to_crs(regions.crs)

charging_stations_with_regions = gpd.sjoin(charging_gdf, regions, how="left", predicate="within")
charging_stations_with_regions = charging_stations_with_regions[charging_stations_with_regions['CNTR_CODE'] == "UK"]
charging_stations_with_regions = charging_stations_with_regions[charging_stations_with_regions["NUTS_NAME"].notna()]
charging_stations_with_regions.columns

Index(['StationID', 'Title', 'AddressLine1', 'Town', 'Latitude', 'Longitude',
       'UsageCost', 'Status', 'PowerKW', 'Voltage', 'Amps', 'CurrentType',
       'ConnectionType', 'Level', 'Quantity', 'geometry', 'index_right',
       'NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME',
       'MOUNT_TYPE', 'URBN_TYPE', 'COAST_TYPE'],
      dtype='object')

## Standardize

In [3]:
charging_stations_with_regions['StationID'] = range(1, len(charging_stations_with_regions) + 1)
charging_stations_with_regions = charging_stations_with_regions.rename(columns={
    'Title': 'chargepoint_name',
    'Longitude': 'x',
    'Latitude': 'y',
    'PowerKW': 'capacity',
    'ConnectionType': 'type',
    'NUTS_NAME': 'RegionName',
})
charging_stations_with_regions = charging_stations_with_regions[['StationID', 'chargepoint_name', 'x', 'y', 'capacity', 'type', 'RegionName', 'NUTS_ID']]
charging_stations_with_regions

,StationID,chargepoint_name,x,y,capacity,type,RegionName,NUTS_ID
0,1,Badger Inn,-2.500894,53.140998,7.0,Type 2 (Socket Only),Cheshire East,UKD62
1,2,"Elmswell Services (EV Tesla OTM) - Elmswell, UK",0.893506,52.231449,250.0,CCS (Type 2),Suffolk,UKH14
2,3,North East Garages,-1.462634,55.054844,22.0,Type 2 (Socket Only),Tyneside,UKC22
3,4,North East Garages,-1.462634,55.054844,22.0,Type 2 (Socket Only),Tyneside,UKC22
4,5,Anglesey Supercharger,-4.264561,53.223435,250.0,CCS (Type 2),Isle of Anglesey,UKL11
...,...,...,...,...,...,...,...,...
47184,47006,Hurst Nissan Centre Belfast,-5.968309,54.571881,NaN,Type 2 (Socket Only),Belfast,UKN06
47185,47007,Alex F Noble & Sons Nissan,-3.173333,55.875053,50.0,CHAdeMO,East Lothian and Midlothian,UKM73
47186,47008,Alex F Noble & Sons Nissan,-3.173333,55.875053,7.0,Type 2 (Socket Only),East Lothian and Midlothian,UKM73
47187,47009,Union Square Car Park,-2.096000,57.142000,7.0,Type 2 (Socket Only),Aberdeen City and Aberdeenshire,UKM50


# Map charging station with buses

In [4]:
df_Buses = pd.read_csv('../../Inputs_prepared/Buses.csv')
df_Buses

,BusID,bus_name,voltage,x,y,RegionName,color
0,1,GB1-220,220.0,-2.895746,58.256815,NaN,red
1,2,GB10-275,275.0,-3.719714,56.088133,Devonside,green
2,3,GB11-275,275.0,-3.179692,55.924620,Whitehouse,red
3,4,GB12-275,275.0,-3.178627,55.903134,Kaimes,red
4,5,GB13-400,400.0,-2.938918,55.884694,Cockenzie,green
...,...,...,...,...,...,...,...
538,539,way/250962683,NaN,-0.258098,51.545075,Added,red
539,540,way/447748328,NaN,-0.136906,51.539316,Added,red
540,541,way/675153143,NaN,0.271833,51.197375,Added,red
541,542,way/1132374248,NaN,-0.136468,51.461563,Added,red


In [5]:
geometry_buses = [Point(xy) for xy in zip(df_Buses['x'], df_Buses['y'])]
buses_gdf = gpd.GeoDataFrame(df_Buses, geometry=geometry_buses, crs="EPSG:4326")

buses_gdf = buses_gdf.to_crs(regions.crs)

buses_with_regions = gpd.sjoin(buses_gdf, regions, how="left", predicate="within")
buses_with_regions.to_csv('./Buses_with_regions.csv', index=False)

In [7]:
charging_stations_with_regions.head()

,StationID,chargepoint_name,x,y,capacity,type,RegionName,NUTS_ID
0,1,Badger Inn,-2.500894,53.140998,7.0,Type 2 (Socket Only),Cheshire East,UKD62
1,2,"Elmswell Services (EV Tesla OTM) - Elmswell, UK",0.893506,52.231449,250.0,CCS (Type 2),Suffolk,UKH14
2,3,North East Garages,-1.462634,55.054844,22.0,Type 2 (Socket Only),Tyneside,UKC22
3,4,North East Garages,-1.462634,55.054844,22.0,Type 2 (Socket Only),Tyneside,UKC22
4,5,Anglesey Supercharger,-4.264561,53.223435,250.0,CCS (Type 2),Isle of Anglesey,UKL11


In [ ]:
from sklearn.neighbors import BallTree
results = []

# Group by regions
for region in charging_stations_with_regions['NUTS_ID'].unique():
    region_stations = charging_stations_with_regions[charging_stations_with_regions['NUTS_ID'] == region]
    region_buses = buses_with_regions[buses_with_regions['NUTS_ID'] == region]

    if region_stations.empty or region_buses.empty:
        continue # Skip if no data for the region

    coords_buses = region_buses[['x', 'y']].to_numpy()
    coords_stations = region_stations[['x', 'y']].to_numpy()


    tree = BallTree(coords_buses, leaf_size=15, metric = 'euclidean')
    distances, indices = tree.query(coords_stations, k=1) # nearest bus only

    for i, idx in enumerate(indices.flatten()):
        result = {
            "charging_id": region_stations.iloc[i]['StationID'],
            "bus_id": region_buses.iloc[idx]['BusID'],
            "distance": distances[i][0],
            "region_ID": region
        }
        results.append(result)
df_regions_ev = pd.DataFrame(results)
df_regions_ev.to_csv('./Regions_EV.csv', index=False)
charging_station_bus_mapping = dict(zip(df_regions_ev['charging_id'], df_regions_ev['bus_id']))
charging_station_bus_mapping

charging_stations_with_regions['BusID'] = charging_stations_with_regions['StationID'].map(charging_station_bus_mapping)
charging_stations_mapped = charging_stations_with_regions
charging_stations_mapped.to_csv('./Charging_Stations.csv', index=False)

# Visualization

In [ ]:
import folium
map_GB = folium.Map(location=[54, -2], zoom_start=6, tiles="CartoDB positron")
regions

,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,geometry
0,HR064,3,HR,Krapinsko-zagorska županija,Krapinsko-zagorska županija,4,3,3,"POLYGON ((15.8824 46.23502, 15.88521 46.23265,..."
1,DE933,3,DE,Harburg,Harburg,4,2,3,"MULTIPOLYGON (((9.78358 53.5, 9.78321 53.49212..."
2,DE934,3,DE,Lüchow-Dannenberg,Lüchow-Dannenberg,4,3,3,"POLYGON ((10.97163 53.1981, 11.01166 53.18148,..."
3,BG314,3,BG,Pleven,Плевен,4,2,3,"POLYGON ((24.6949 43.69762, 24.69783 43.69603,..."
4,BG331,3,BG,Varna,Варна,4,2,1,"POLYGON ((27.42946 43.60027, 27.4397 43.58454,..."
...,...,...,...,...,...,...,...,...,...
1509,UKG37,3,UK,Sandwell,Sandwell,4,1,3,"POLYGON ((-1.98893 52.56297, -1.97521 52.55595..."
1510,UKM77,3,UK,Perth & Kinross and Stirling,Perth & Kinross and Stirling,2,2,1,"MULTIPOLYGON (((-3.90435 56.93136, -3.89569 56..."
1511,NO0B1,3,NO,Jan Mayen,Jan Mayen,3,3,1,"POLYGON ((-9.08295 70.86469, -9.07843 70.86831..."
1512,EE009,3,EE,Kesk-Eesti,Kesk-Eesti,4,3,1,"MULTIPOLYGON (((25.98343 59.62115, 25.99837 59..."


In [ ]:
import json

# 1. Load NUTS level-3 GeoJSON
# with open("./ref-nuts-2021-01m.geojson/NUTS_RG_01M_2021_4326_LEVL_3.geojson","r", encoding="utf-8") as f:
#     regions = json.load(f)

import geopandas as gpd

regions = gpd.read_file("./ref-nuts-2021-01m.geojson/NUTS_RG_01M_2021_4326_LEVL_3.geojson")
regions = regions[regions["CNTR_CODE"] == "UK"]

# Add regions to map
folium.GeoJson(
    regions,
    name="UK NUTS2 Regions",
    style_function=lambda feature: {
        "color": "#888888",
        "weight": 1,
    },
    tooltip=folium.GeoJsonTooltip(fields=["NUTS_NAME"], aliases=["Region:"])
).add_to(map_GB)



# Highlight those regions without buses

In [ ]:

regions_with_buses = buses_with_regions['index_right'].dropna().unique()
regions['has_bus'] = regions.index.isin(regions_with_buses)
def style_function(feature):
    if feature["properties"]["has_bus"]:
        return {
            "color": "#888888",
            "weight": 1,
            "fillOpacity": 0.1
        }
    else:
        return {
            "color": "#1f77b4",  # Highlight regions with no buses
            "weight": 2,
            "fillOpacity": 0.4
        }

# Convert regions GeoDataFrame to GeoJSON for folium
regions_json = json.loads(regions.to_json())

# Add styled GeoJson to folium map
folium.GeoJson(
    data=regions_json,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["NUTS_NAME"], aliases=["Region:"])
).add_to(map_GB)


# Added Lines between buses

In [ ]:
df_lines = pd.read_csv('/Users/zm348/PhD/Projects/Nature-EV/data/Inputs_prepared/Lines.csv')

x = dict(zip(df_Buses['bus_name'], df_Buses['x']))
y = dict(zip(df_Buses['bus_name'], df_Buses['y']))

for _, row in df_lines.iterrows():
    bus0, bus1 = row['bus0'], row['bus1']
    
    try:
        coord0 = (y[bus0], x[bus0])
        coord1 = (y[bus1], x[bus1])

        color = 'orange' if row['length'] == "Added" else 'blue'

        folium.PolyLine(
            locations=[coord0, coord1],
            color=color,
            weight=1.2,
            opacity=0.8,
            popup=f"{bus0} → {bus1}"
        ).add_to(map_GB)
    
    except KeyError as e:
        print(f"Missing bus in coordinate mapping: {e}")
        continue

Missing bus in coordinate mapping: 'way/264275258-275'
Missing bus in coordinate mapping: 'way/264275258-275'


# Add charging stations

In [ ]:
for _, row in charging_stations_with_regions.iterrows():
    if pd.isna(row["Latitude"]) or pd.isna(row["Longitude"]):
        continue  # skip rows with missing location
    popup_html = f"""
    <b>charging station:</b> {row.get("Title", "")}<br>
    <b>x:</b> {row.get("Longitude", "")}<br>
    <b>y:</b> {row.get("Latitude", "")}<br>
    <b>Capacity:</b> {row.get("PowerKW", "")}
    """

    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=2,
        color="purple",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=200)
    ).add_to(map_GB)

In [ ]:
df_Buses

,BusID,bus_name,voltage,x,y,RegionName,color
0,1,GB1-220,220.0,-2.895746,58.256815,NaN,red
1,2,GB10-275,275.0,-3.719714,56.088133,Devonside,green
2,3,GB11-275,275.0,-3.179692,55.924620,Whitehouse,red
3,4,GB12-275,275.0,-3.178627,55.903134,Kaimes,red
4,5,GB13-400,400.0,-2.938918,55.884694,Cockenzie,green
...,...,...,...,...,...,...,...
538,539,way/250962683,NaN,-0.258098,51.545075,Added,red
539,540,way/447748328,NaN,-0.136906,51.539316,Added,red
540,541,way/675153143,NaN,0.271833,51.197375,Added,red
541,542,way/1132374248,NaN,-0.136468,51.461563,Added,red


# Add buses

In [ ]:
for _, row in df_Buses.iterrows():

    popup_html = f"""
    <b>BusID:</b> {row.get("BusID", "")}<br>
    <b>Bus:</b> {row.get("bus_name", "")}<br>
    <b>x:</b> {row.get("x", "")}<br>
    <b>y:</b> {row.get("y", "")}<br>
    <b>Capacity:</b> {row.get("PowerKW", "")}
    """

    folium.CircleMarker(
        location=[row["y"], row["x"]],
        radius=2,
        color="red",
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_html, max_width=200)
    ).add_to(map_GB)

In [ ]:
map_GB.save("Charging_Station_Visualization.html")
